In [1]:
!pip install -q --upgrade \
    bitsandbytes==0.48.2 \
    trl==0.25.1 \
    transformers==4.57.6 \
    peft \
    datasets \
    accelerate \
    huggingface_hub

In [2]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from datasets import load_dataset

from peft import LoraConfig

from trl import SFTTrainer, SFTConfig

In [3]:
# -------------------------
# Model / Project
# -------------------------

BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "medical-research-assistant"

# Change this to your Hugging Face username
HF_USER = "YOUR_HF_USERNAME"


# -------------------------
# Fine-tuning settings
# -------------------------

EPOCHS = 2

BATCH_SIZE = 4

MAX_SEQUENCE_LENGTH = 512

GRADIENT_ACCUMULATION_STEPS = 4


# -------------------------
# QLoRA settings
# -------------------------

QUANT_4_BIT = True

LORA_R = 16

LORA_ALPHA = 32

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
]

LORA_DROPOUT = 0.05


# -------------------------
# Training settings
# -------------------------

LEARNING_RATE = 2e-4

WARMUP_RATIO = 0.03

LR_SCHEDULER_TYPE = "cosine"

WEIGHT_DECAY = 0.01

OPTIMIZER = "paged_adamw_32bit"


# -------------------------
# GPU settings
# -------------------------

capability = torch.cuda.get_device_capability()

use_bf16 = capability[0] >= 8

print("GPU capability:", capability)
print("Using BF16:", use_bf16)


# -------------------------
# Logging / evaluation
# -------------------------

LOG_STEPS = 10

SAVE_STEPS = 100

LOG_TO_WANDB = False

GPU capability: (7, 5)
Using BF16: False


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

PyTorch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [6]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

login(
    token=hf_token,
    add_to_git_credential=True
)

print("Logged in to Hugging Face successfully.")

Logged in to Hugging Face successfully.


In [7]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

In [11]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

from datasets import load_dataset

RuntimeError: Failed to import trl.trainer.sft_trainer because of the following error (look up to see its traceback):
cannot import name 'download_url' from 'transformers.utils' (/usr/local/lib/python3.13/dist-packages/transformers/utils/__init__.py)

In [8]:
# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


# Load model

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

base_model.generation_config.pad_token_id = tokenizer.pad_token_id


print(
    f"Memory footprint: "
    f"{base_model.get_memory_footprint() / 1e6:.1f} MB"
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 2197.6 MB


In [9]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
]

In [10]:
from peft import LoraConfig

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [11]:
# Training parameters

train_parameters = SFTConfig(
    output_dir="medical_rag_finetuned",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    weight_decay=0.001,

    fp16=not use_bf16,
    bf16=use_bf16,

    max_grad_norm=0.3,

    warmup_ratio=0.03,

    lr_scheduler_type="cosine",

    logging_steps=10,

    save_strategy="epoch",

    eval_strategy="epoch",

    report_to="none",

    max_length=512,

    packing=False,
)

In [12]:
from datasets import Dataset, DatasetDict

medical_data = [
    {
        "question": "What are the limitations of medical RAG systems?",
        "answer": "Medical RAG systems can struggle with complex integration tasks, noise, misinformation, and robustness across different practical scenarios."
    },
    {
        "question": "Why is retrieval important in a medical RAG system?",
        "answer": "Retrieval provides relevant information from external medical documents so that the language model can use that information when generating an answer."
    },
    {
        "question": "What is the purpose of a medical research assistant?",
        "answer": "A medical research assistant helps users retrieve and understand information from medical documents while grounding its responses in the available evidence."
    },
    {
        "question": "What is RAG?",
        "answer": "RAG, or Retrieval-Augmented Generation, combines information retrieval with language generation. Relevant documents are retrieved and provided to a language model as context for generating an answer."
    },
    {
        "question": "Why can medical RAG systems require specialized components?",
        "answer": "Specialized components can complement the strengths of language models and help mitigate weaknesses such as difficulty with complex integration tasks, noise, and misinformation."
    },
]

dataset = Dataset.from_list(medical_data)

dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 5
})

In [13]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train = dataset["train"]
val = dataset["test"]

print("Training examples:", len(train))
print("Validation examples:", len(val))

Training examples: 4
Validation examples: 1


In [14]:
def format_example(example):
    return {
        "text": f"""### Question:
{example["question"]}

### Answer:
{example["answer"]}"""
    }

train = train.map(format_example)
val = val.map(format_example)

print(train[0])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

{'question': 'What is the purpose of a medical research assistant?', 'answer': 'A medical research assistant helps users retrieve and understand information from medical documents while grounding its responses in the available evidence.', 'text': '### Question:\nWhat is the purpose of a medical research assistant?\n\n### Answer:\nA medical research assistant helps users retrieve and understand information from medical documents while grounding its responses in the available evidence.'}


In [15]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

In [16]:
fine_tuning.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,No log,4.011330,3.430946,171.000000,0.318182
2,No log,3.837630,3.396467,342.000000,0.363636


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=2, training_loss=3.174710750579834, metrics={'train_runtime': 12.4767, 'train_samples_per_second': 0.641, 'train_steps_per_second': 0.16, 'total_flos': 6108312453120.0, 'train_loss': 3.174710750579834, 'epoch': 2.0})

In [17]:
fine_tuning.save_model("medical_llama_lora")